# Problema de la mochila : Algoritmo Genético
Hugo André Meza Fierros

A00841695

# Problema 1

Un armador tiene un carguero con capacidad de hasta 800 toneladas. El carguero transporta contenedores de diferentes pesos para una determinada ruta. En la ruta actual el carguero puede transportar algunos de los siguientes contenedores:
| Contenedor | $c_1$ | $c_2$ | $c_3$ | $c_4$ | $c_5$ | $c_6$ | $c_7$ | $c_8$ | $c_9$ | $c_{10}$ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Peso [ton]** | 61 | 58 | 92 | 50 | 108 | 83 | 93 | 101 | 54 | 50 |
| **Beneficio [\$]** | 1100 | 1147 | 1442 | 1591 | 1078 | 1385 | 1777 | 1196 | 1753 | 1371 |

<br>

| Contenedor | $c_{11}$ | $c_{12}$ | $c_{13}$ | $c_{14}$ | $c_{15}$ | $c_{16}$ | $c_{17}$ | $c_{18}$ | $c_{19}$ | $c_{20}$ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Peso [ton]** | 72 | 51 | 100 | 108 | 91 | 112 | 66 | 58 | 110 | 73 |
| **Beneficio [\$]** | 1517 | 1675 | 1193 | 1177 | 1365 | 1143 | 1314 | 1526 | 1470 | 1605 |

El analista de la empresa del armador desea determinar el envío (conjunto de contenedores) que maximiza el beneficio.

## Método exacto (CPLEX)

In [2]:
from docplex.mp.model import Model
T = 800
I = range(1, 21)
w = [61, 58, 92, 50, 108, 83, 93, 101, 54, 50, 72, 51, 100, 108, 91, 112, 66, 58, 110, 73]
c = [1100, 1147, 1442, 1591, 1078, 1385, 1777, 1196, 1753, 1371, 1517, 1675, 1193, 1177, 1365, 1143, 1314, 1526, 1470, 1605]

In [3]:
mdl = Model('Mochila')

x = mdl.binary_var_list(I, name='x') # variable de decisión

mdl.maximize(mdl.sum(c[i-1] * x[i-1] for i in I)) # Función objetivo

mdl.add_constraint(mdl.sum(w[i-1] * x[i-1] for i in I) <= T, ctname='r_capacidad') # Restricción de capacidad

docplex.mp.LinearConstraint[r_capacidad](61x_1+58x_2+92x_3+50x_4+108x_5+83x_6+93x_7+101x_8+54x_9+50x_10+72x_11+51x_12+100x_13+108x_14+91x_15+112x_16+66x_17+58x_18+110x_19+73x_20,LE,800)

In [4]:
solucion = mdl.solve()

if solucion:
    # 1. Valor de la función objetivo Z
    valor_objetivo = solucion.objective_value
    
    # Obtener el objeto de detalles del solver
    detalles = mdl.get_solve_details()
    
    # 2. Tiempo computacional y número de iteraciones
    tiempo_cpu = detalles.time
    num_iteraciones = detalles.nb_iterations

    # 3. Solución factible (Contenedores donde x_i = 1)
    # x[i].solution_value da el valor asignado a la variable en la solución
    seleccionados = [i for i in I if x[i-1].solution_value > 0.5]

    # 4. Capacidad ocupada (Suma de pesos de los contenedores seleccionados)
    capacidad_ocupada = sum(w[i - 1] for i in seleccionados)
    porcentaje_ocupacion = (capacidad_ocupada / T) * 100

    # Imprimir reporte completo
    print("=" * 45)
    print("       REPORTE DE MÉTRICAS DE EJECUCIÓN (CPLEX)")
    print("=" * 45)
    print(f"Función Objetivo (Z) : ${valor_objetivo:,.2f}")
    print(f"Solución Factible (S): Contenedores {seleccionados}")
    print(f"Capacidad Ocupada    : {capacidad_ocupada} / {T} ton ({porcentaje_ocupacion:.1f}%)")
    print(f"Tiempo Computacional : {tiempo_cpu:.6f} segundos")
    print(f"Número de Iteraciones: {num_iteraciones}")
    print("=" * 45)
else:
    print("No se encontró una solución factible.")

       REPORTE DE MÉTRICAS DE EJECUCIÓN (CPLEX)
Función Objetivo (Z) : $18,103.00
Solución Factible (S): Contenedores [2, 3, 4, 6, 7, 9, 10, 11, 12, 17, 18, 20]
Capacidad Ocupada    : 800 / 800 ton (100.0%)
Tiempo Computacional : 0.172000 segundos
Número de Iteraciones: 2


## Algoritmo genético

In [24]:
import numpy as np
import time

def objective_function(c, w, T, vector):
    total_value = np.sum(c * vector)
    total_weight = np.sum(w * vector)
    return total_value, total_weight

def fitness_function(c, w, T, vector, variante):
    total_value, total_weight = objective_function(c, w, T, vector)
    if total_weight > T:
        if variante == 1:
            return total_value / 2.0  # Variante 1: Permite infactibilidad (penaliza)
        else:
            return 0.0                # Variante 2: Solo factibles (rechaza por completo)
    else:
        return total_value

def ruleta(fitness_values):
    fitness_values = np.maximum(fitness_values, 0) # Evitar fitness negativos
    total_fitness = np.sum(fitness_values)
    
    if total_fitness == 0:
        probabilities = np.ones(len(fitness_values)) / len(fitness_values)
    else:
        probabilities = fitness_values / total_fitness
        
    padres = np.random.choice(len(fitness_values), size=2, p=probabilities, replace=False)
    return padres[0], padres[1]

def cruza_mitad(padre1, padre2):
    """Cruza exactamente a la mitad (Lugar de cruce: n/2)"""
    mitad = len(padre1) // 2
    hijo1 = np.concatenate((padre1[:mitad], padre2[mitad:]))
    hijo2 = np.concatenate((padre2[:mitad], padre1[mitad:]))
    return hijo1, hijo2

def mutacion(hijo, tasa_mutacion=0.05):
    hijo_mutado = np.copy(hijo)
    for i in range(len(hijo_mutado)):
        if np.random.rand() < tasa_mutacion:
            hijo_mutado[i] = 1 - hijo_mutado[i]
    return hijo_mutado

In [48]:
def ejecutar_genetico(c, w, T, num_vectores=12, num_iteraciones=1500, variante=1):
    n = len(c)
    tasa_mutacion = 0.05  # Probabilidad de mutación
    
    # Inicialización Aleatoria Uniforme
    poblacion_actual = np.random.randint(0, 2, size=(num_vectores, n))
    
    # Evaluar la población inicial
    fitness_poblacion = np.array([fitness_function(c, w, T, ind, variante) for ind in poblacion_actual])

    historial_mejor_fitness = []

    # Iniciar cronómetro
    start_time = time.time()
    
    # Ciclo de Estado Estacionario
    for iteracion in range(num_iteraciones):
        
        # 1. Selección (Ruleta)
        idx_p1, idx_p2 = ruleta(fitness_poblacion)
        padre1 = poblacion_actual[idx_p1]
        padre2 = poblacion_actual[idx_p2]
        
        # 2. Cruza (1 punto a la mitad)
        hijo1, hijo2 = cruza_mitad(padre1, padre2)
        
        # 3. Mutación
        hijo1 = mutacion(hijo1, tasa_mutacion)
        hijo2 = mutacion(hijo2, tasa_mutacion)
        
        # 4. Evaluar hijos
        fit_h1 = fitness_function(c, w, T, hijo1, variante)
        fit_h2 = fitness_function(c, w, T, hijo2, variante)
        
        # 5. REEMPLAZO: Encontrar los 2 peores individuos
        # argsort ordena de menor a mayor fitness, tomamos los 2 primeros índices
        indices_peores = np.argsort(fitness_poblacion)[:2]
        
        # Sustituir a los 2 peores con los nuevos hijos
        poblacion_actual[indices_peores[0]] = hijo1
        fitness_poblacion[indices_peores[0]] = fit_h1
        
        poblacion_actual[indices_peores[1]] = hijo2
        fitness_poblacion[indices_peores[1]] = fit_h2

        mejor_actual = np.max(fitness_poblacion)
        historial_mejor_fitness.append(mejor_actual)
        
    # Detener cronómetro
    end_time = time.time()
    tiempo_cpu = end_time - start_time
    
    # Buscar el mejor individuo histórico que quedó en la población final
    mejor_idx = np.argmax(fitness_poblacion)
    mejor_vector = poblacion_actual[mejor_idx]
    mejor_fitness = fitness_poblacion[mejor_idx]
    mejor_valor, mejor_peso = objective_function(c, w, T, mejor_vector)
    
    return (mejor_vector, mejor_valor, mejor_peso, mejor_fitness, tiempo_cpu, 
            poblacion_actual, fitness_poblacion, historial_mejor_fitness)

In [49]:
z_optimo_cplex = valor_objetivo 

print("=== VARIANTE 1: PERMITE INFACTIBILIDAD (Penaliza a la mitad) ===")
(vec1, val1, peso1, fit1, tiempo1, 
 pob_final1, fit_pob_final1, historial1) = ejecutar_genetico(c, w, T, variante=1)
brecha1 = ((z_optimo_cplex - val1) / z_optimo_cplex) * 100
print(f"Mejor Valor: {val1} | Peso: {peso1} | Tiempo CPU: {tiempo1:.4f}s | Diferencia vs Óptimo: {brecha1:.2f}%")

print("\n=== VARIANTE 2: SOLO FACTIBLES (Fitness 0 si incumple) ===")
(vec2, val2, peso2, fit2, tiempo2, 
 pob_final2, fit_pob_final2, historial2) = ejecutar_genetico(c, w, T, variante=2)
brecha2 = ((z_optimo_cplex - val2) / z_optimo_cplex) * 100
print(f"Mejor Valor: {val2} | Peso: {peso2} | Tiempo CPU: {tiempo2:.4f}s | Diferencia vs Óptimo: {brecha2:.2f}%")

=== VARIANTE 1: PERMITE INFACTIBILIDAD (Penaliza a la mitad) ===
Mejor Valor: 17092 | Peso: 794 | Tiempo CPU: 0.0731s | Diferencia vs Óptimo: 5.58%

=== VARIANTE 2: SOLO FACTIBLES (Fitness 0 si incumple) ===
Mejor Valor: 17846 | Peso: 796 | Tiempo CPU: 0.0701s | Diferencia vs Óptimo: 1.42%


In [50]:

print("\n--- POBLACIÓN FINAL (VARIANTE 1) ---")
for i in range(len(pob_final1)):
    print(f"Individuo {i+1}: {pob_final1[i]} | Fitness: {fit_pob_final1[i]}")

print("\n--- POBLACIÓN FINAL (VARIANTE 2) ---")
for i in range(len(pob_final2)):
    print(f"Individuo {i+1}: {pob_final2[i]} | Fitness: {fit_pob_final2[i]}")



--- POBLACIÓN FINAL (VARIANTE 1) ---
Individuo 1: [0 0 1 1 0 0 1 0 1 1 1 0 0 0 1 0 0 1 1 1] | Fitness: 15417.0
Individuo 2: [0 0 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 13897.0
Individuo 3: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 4: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 5: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 6: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 7: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 8: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 9: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 10: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 11: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0
Individuo 12: [0 0 1 1 0 0 1 0 1 1 1 1 0 0 1 0 0 1 1 1] | Fitness: 17092.0

--- POBLACIÓN FINAL (VARIANTE 2) ---
Individuo 1: [1 1 0 1 0 0 0 0 1 1